# 08 · Tilt, phase ramps, and optical masks: build it step by step

A linear phase ramp tilts a beam by shifting its propagation direction. This is the simplest phase mask and a useful way to connect spatial phase to an observable displacement.

This companion develops the same idea with small functions defined beside the experiment that needs them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Define the phase ramp

The ramp slope is set by transverse wave number `k sin(α)`. This function applies only phase, keeping the beam envelope intact.

In [ ]:
def tilt_phase(x, wavelength_m, angle_rad):
    """Complex phase mask for a plane wave tilted in x."""
    k = 2*np.pi/wavelength_m
    return np.exp(1j*k*x*np.sin(angle_rad))

lambda_m = 633e-9
window_m = 8e-3
n = 2048
x_m = np.linspace(-window_m/2, window_m/2, n, endpoint=False)
field0 = np.exp(-x_m**2/(2*(1e-3)**2))
angle_rad = np.deg2rad(0.3)
field_tilted = field0*tilt_phase(x_m, lambda_m, angle_rad)


## 2. Define propagation for the tilted field

The phase ramp changes direction; propagation converts that direction into displacement at the observation plane.

In [ ]:
def propTF(field, window_m, wavelength_m, z_m):
    """Paraxial Fresnel propagation of a 1D field."""
    n = field.size
    dx = window_m/n
    freq = np.fft.fftshift(np.fft.fftfreq(n, d=dx))
    transfer = np.exp(-1j*np.pi*wavelength_m*z_m*freq**2)
    spectrum = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(field)))
    return np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(spectrum*transfer)))


## 2. Connect phase to beam displacement

Propagation turns the phase slope into a lateral shift. The intensity centroid is the first moment of the output distribution.

In [ ]:
z_m = 0.30
field_z = propTF(field_tilted, window_m, lambda_m, z_m)
I = np.abs(field_z)**2
center_m = np.sum(x_m*I)/np.sum(I)
print(f"Measured centroid: {center_m*1e3:.3f} mm")
print(f"Small-angle estimate: {z_m*np.tan(angle_rad)*1e3:.3f} mm")


## Figures

No matching plot exists in the paper notebooks for this phase-ramp topic, so this lesson contains no plot.
